# Article eligibility and user profiles

BT4222 Group 6 · representations 3/3. Run both article notebooks first.

Build observation-time article eligibility, supplied-history profiles for baselines/clustering, and demonstrate
causal profiles with long-term, recent and up to three category-interest components. Actual request-time
history references are exported by the feature notebook during chronological replay.

Run top to bottom using the same Python environment as the other notebooks. All project code and settings are included below; no local Python module or settings file is imported. Upstream Parquet/NumPy artifacts remain the inputs shared between notebooks.

`REBUILD=False` reuses verified saved outputs; set it to `True` after changing this notebook's calculations/settings. Restarting VS Code clears variables, not saved datasets.

In [ ]:
# All project settings and implementation are in this notebook.
# Standard third-party packages come from requirements.txt.
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import google.colab
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass

# Edit DATA_ROOT to point to another project/data folder, if necessary.
DATA_ROOT = None
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/BT4222")]
if DATA_ROOT is not None:
    ROOT = Path(DATA_ROOT).expanduser().resolve()
elif os.environ.get("NEWSREC_ROOT"):
    ROOT = Path(os.environ["NEWSREC_ROOT"]).expanduser().resolve()
else:
    ROOT = next(
        (
            p
            for p in candidates
            if (p / "1_Data_Preparation/processed/news_catalogue.parquet").is_file()
        ),
        None,
    )
if ROOT is None:
    raise FileNotFoundError("Run data preparation first, or set DATA_ROOT above.")
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data and output root:", ROOT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
from collections import Counter
import polars as pl
import sqlite3
from collections import Counter, deque
from dataclasses import dataclass, field
import heapq
import pyarrow.parquet as pq

### Artifact checks and reporting

In [ ]:
NOTEBOOK_PATHS = (
    "3_Representations/news_embeddings.ipynb",
    "3_Representations/entity_embeddings.ipynb",
    "3_Representations/user_profiles.ipynb",
    "4_Feature_Engineering/build_ranking_features.ipynb",
)

In [ ]:
SPLITS = ("train", "validation", "test")

In [ ]:
VERSION = "newsrec-v1"

In [ ]:
@dataclass(frozen=True)
class Settings:
    seed: int = 4222
    text_dim: int = 64
    max_features: int = 50000
    min_df: int = 2
    title_weight: float = 0.7
    title_tokens: int = 30
    entity_dim: int = 100
    history_cap: int = 50
    snapshot_order_verified: bool = False
    recent_count: int = 10
    recent_hours: int = 24
    interest_count: int = 3
    ctr_strength: float = 20.0
    half_life_min_exposures: int = 100
    negative_count: int = 4
    update_mode: str = "prequential"
    context_features: bool = False
    parquet_batch_rows: int = 100000
    output_batch_rows: int = 100000

    def __post_init__(self):
        for key in (
            "text_dim",
            "max_features",
            "min_df",
            "title_tokens",
            "entity_dim",
            "history_cap",
            "recent_count",
            "recent_hours",
            "interest_count",
            "half_life_min_exposures",
            "negative_count",
            "parquet_batch_rows",
            "output_batch_rows",
        ):
            if getattr(self, key) <= 0:
                raise ValueError(f"{key} must be positive")
        if not 0 <= self.title_weight <= 1 or self.ctr_strength <= 0:
            raise ValueError("Invalid fusion/smoothing weights")
        if self.update_mode not in ("prequential", "frozen"):
            raise ValueError("update_mode must be prequential or frozen")

    def representation_fingerprint(self):
        fields = (
            "seed",
            "text_dim",
            "max_features",
            "min_df",
            "title_weight",
            "title_tokens",
            "entity_dim",
        )
        values = {name: getattr(self, name) for name in fields}
        return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()

    def fingerprint(self):
        return hashlib.sha256(
            json.dumps(asdict(self), sort_keys=True).encode()
        ).hexdigest()

In [ ]:
DEFAULT_SETTINGS = Settings()

In [ ]:
# Edit settings here. Keep representation dimensions/vocabulary settings
# consistent across the notebooks that consume those artifacts.
SETTINGS = Settings(
    seed=4222,
    text_dim=64,
    max_features=50000,
    min_df=2,
    title_weight=0.7,
    title_tokens=30,
    entity_dim=100,
    history_cap=50,
    snapshot_order_verified=False,
    recent_count=10,
    recent_hours=24,
    interest_count=3,
    ctr_strength=20.0,
    half_life_min_exposures=100,
    negative_count=4,
    update_mode="prequential",
    context_features=False,
    parquet_batch_rows=100000,
    output_batch_rows=100000,
)
REBUILD = (
    False  # False: reuse verified outputs; True: regenerate this notebook's outputs
)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n")

In [ ]:
def normalise(values):
    values = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(values, axis=-1, keepdims=True)
    return np.divide(values, norms, out=np.zeros_like(values), where=norms > 1e-12)

In [ ]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))

In [ ]:
def manifest(root, cfg, sources, outputs, **extra):
    root = Path(root)
    try:
        revision = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        revision = None
    return dict(
        version=VERSION,
        created_utc=pd.Timestamp.now(tz="UTC").isoformat(),
        config=asdict(cfg),
        config_hash=cfg.fingerprint(),
        representation_config_hash=cfg.representation_fingerprint(),
        git_revision=revision,
        python=platform.python_version(),
        libraries={
            n: importlib.metadata.version(n)
            for n in ("numpy", "pandas", "pyarrow", "polars", "scikit-learn")
        },
        implementation_hashes={
            relative: sha256(root / relative)
            for relative in NOTEBOOK_PATHS
            if (root / relative).is_file()
        },
        sources={str(Path(p).relative_to(root)): sha256(p) for p in sources},
        outputs={str(Path(p).relative_to(root)): sha256(p) for p in outputs},
        output_bytes=sum((Path(p).stat().st_size for p in outputs)),
        **extra,
    )

In [ ]:
@contextmanager
def measure(root, stage, **details):
    import threading

    process = psutil.Process()
    peak = [process.memory_info().rss]
    stop = threading.Event()

    def monitor():
        while not stop.wait(0.05):
            peak[0] = max(peak[0], process.memory_info().rss)

    worker = threading.Thread(target=monitor, daemon=True)
    worker.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        peak[0] = max(peak[0], process.memory_info().rss)
        stop.set()
        worker.join()
        path = Path(root) / "results" / "representation_cost.csv"
        path.parent.mkdir(parents=True, exist_ok=True)
        row = dict(
            stage=stage,
            wall_seconds=time.perf_counter() - start,
            sampled_peak_process_rss_mb=peak[0] / 1000000.0,
            platform=platform.platform(),
            cpu=platform.processor() or platform.machine(),
            logical_cpus=psutil.cpu_count(),
            ram_gb=psutil.virtual_memory().total / 1000000000.0,
            **details,
        )
        pd.DataFrame([row]).to_csv(
            path, mode="a", header=not path.exists(), index=False
        )

In [ ]:
def upsert_report(root, filename, frame, stage):
    path = Path(root) / "results" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = frame.copy()
    frame["stage"] = stage
    if path.exists():
        old = pd.read_csv(path)
        frame = pd.concat([old[old.stage != stage], frame], ignore_index=True)
    frame.to_csv(path, index=False)

In [ ]:
def require_inputs(root):
    p = Path(root) / "1_Data_Preparation" / "processed"
    files = [p / "news_catalogue.parquet", p / "users.parquet"]
    files += [
        p / f"{kind}_{split}.parquet"
        for kind in ("impressions", "candidates")
        for split in SPLITS
    ]
    missing = [str(f) for f in files if not f.exists()]
    if missing:
        raise FileNotFoundError(
            "Run 1_Data_Preparation/clean_and_validate_data.ipynb first. Missing:\n"
            + "\n".join(missing)
        )
    return p

### Article representation and availability logic

In [ ]:
def artifact_dir(root):
    path = Path(root) / "3_Representations" / "artifacts"
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
def build_availability(root, cfg=DEFAULT_SETTINGS):
    """Discovery from earliest observable history/slate, never an invented publication date."""
    processed = require_inputs(root)
    out = artifact_dir(root)
    with measure(root, "availability"):
        histories = []
        exposures = []
        for split in SPLITS:
            histories.append(
                pl.scan_parquet(processed / f"impressions_{split}.parquet").select(
                    "user_code", "timestamp", "history"
                )
            )
            exposures.append(
                pl.scan_parquet(processed / f"candidates_{split}.parquet").select(
                    "news_code", "timestamp"
                )
            )
        first_hist = (
            pl.concat(histories)
            .group_by("user_code", "history")
            .agg(pl.col("timestamp").min())
            .filter(pl.col("history").is_not_null() & (pl.col("history") != ""))
            .with_columns(pl.col("history").str.split(" ").alias("news_id"))
            .explode("news_id", empty_as_null=True)
            .with_columns(
                pl.col("news_id")
                .str.strip_prefix("N")
                .cast(pl.Int32)
                .alias("news_code")
            )
            .group_by("news_code")
            .agg(pl.col("timestamp").min().alias("history_discovered_at"))
        )
        first_exp = (
            pl.concat(exposures)
            .group_by("news_code")
            .agg(pl.col("timestamp").min().alias("first_observed_exposure"))
        )
        frame = (
            first_hist.join(first_exp, on="news_code", how="full", coalesce=True)
            .with_columns(
                pl.min_horizontal(
                    "history_discovered_at", "first_observed_exposure"
                ).alias("available_from")
            )
            .sort("news_code")
            .collect(engine="streaming")
            .to_pandas()
        )
        frame["discovery_source"] = np.where(
            frame.history_discovered_at.isna(),
            "slate",
            np.where(
                frame.first_observed_exposure.isna()
                | (frame.history_discovered_at < frame.first_observed_exposure),
                "history",
                "slate_or_tied",
            ),
        )
        start = min(
            (
                pl.scan_parquet(processed / f"impressions_{s}.parquet")
                .select(pl.col("timestamp").min())
                .collect()
                .item()
                for s in SPLITS
            )
        )
        frame["left_censored"] = frame.first_observed_exposure.eq(start)
        frame.to_parquet(out / "article_availability.parquet", index=False)
        if (out / "article_index.parquet").exists() and (
            out / "article_entity100.npy"
        ).exists():
            index = pd.read_parquet(out / "article_index.parquet").set_index(
                "news_code"
            )
            audit = frame.sort_values("available_from").copy()
            ordered_rows = index.loc[audit.news_code, "article_row"].to_numpy(np.int32)
            text_valid = (
                np.linalg.norm(
                    np.load(out / "text64.npy", mmap_mode="r")[ordered_rows], axis=1
                )
                > 0
            )
            entity_valid = (
                np.linalg.norm(
                    np.load(out / "article_entity100.npy", mmap_mode="r")[ordered_rows],
                    axis=1,
                )
                > 0
            )
            text_cum = np.r_[0, np.cumsum(text_valid)]
            entity_cum = np.r_[0, np.cumsum(entity_valid)]
            times = pd.to_datetime(audit.available_from).to_numpy(
                dtype="datetime64[ns]"
            )
            checkpoints = pd.date_range(
                pd.Timestamp(start).floor("h"),
                pd.Timestamp(frame.available_from.max()).ceil("h"),
                freq="h",
            )
            counts = np.searchsorted(
                times, checkpoints.to_numpy(dtype="datetime64[ns]"), side="right"
            )
            report = pd.DataFrame(
                {
                    "timestamp": checkpoints,
                    "eligible_articles": counts,
                    "eligible_text_vectors": text_cum[counts],
                    "eligible_entity_vectors": entity_cum[counts],
                }
            )
            report_path = Path(root) / "results" / "catalogue_coverage_over_time.csv"
            report_path.parent.mkdir(parents=True, exist_ok=True)
            report.to_csv(report_path, index=False)
        sources = [
            processed / f"{kind}_{s}.parquet"
            for kind in ("impressions", "candidates")
            for s in SPLITS
        ]
        write_json(
            out / "availability_manifest.json",
            manifest(
                root,
                cfg,
                sources,
                [out / "article_availability.parquet"],
                policy="minimum observable history/slate timestamp <= request; not publication time",
            ),
        )
    return frame

In [ ]:
class Bundle:
    """Validated article-aligned arrays. Row 0 is padding/unknown, never a real article."""

    def __init__(self, root, cfg=DEFAULT_SETTINGS, verify_hashes=True):
        self.root = Path(root)
        self.cfg = cfg
        out = artifact_dir(root)
        index_hash = sha256(out / "article_index.parquet")
        for name in ("news", "entity", "availability"):
            meta = json.loads((out / f"{name}_manifest.json").read_text())
            if meta["representation_config_hash"] != cfg.representation_fingerprint():
                raise ValueError(
                    f"{name}: incompatible configuration; rebuild artifacts"
                )
            if meta.get("article_index_hash", index_hash) != index_hash:
                raise ValueError(
                    f"{name}: article map provenance mismatch; rebuild dependents"
                )
            if verify_hashes:
                for rel, expected in meta["outputs"].items():
                    if sha256(self.root / rel) != expected:
                        raise ValueError(f"Artifact hash mismatch: {rel}")
        self.index = pd.read_parquet(out / "article_index.parquet")
        self.lookup = dict(zip(self.index.news_code, self.index.article_row))
        self.n = len(self.index) + 1
        self.title = np.load(out / "title64.npy", mmap_mode="r")
        self.text = np.load(out / "text64.npy", mmap_mode="r")
        self.entity = np.load(out / "article_entity100.npy", mmap_mode="r")
        for array, dim in (
            (self.title, cfg.text_dim),
            (self.text, cfg.text_dim),
            (self.entity, cfg.entity_dim),
        ):
            norms = np.linalg.norm(array, axis=1)
            if (
                array.shape != (self.n, dim)
                or not np.isfinite(array).all()
                or np.any((norms > 1e-07) & (np.abs(norms - 1) > 0.0001))
                or (norms[0] != 0)
            ):
                raise ValueError(
                    "Invalid representation shape, coordinates, norm or padding"
                )
        self.meta = self.index.set_index("article_row").reindex(range(self.n))
        self.quality = pd.read_parquet(out / "entity_quality.parquet").set_index(
            "news_code"
        )
        self.availability = pd.read_parquet(
            out / "article_availability.parquet"
        ).set_index("news_code")
        if set(self.lookup) != set(self.availability.index):
            raise ValueError("Availability must cover exactly the prepared catalogue")
        self.category = np.r_[0, self.index.category_code.to_numpy(np.int32)]
        self.subcategory = np.r_[0, self.index.subcategory_code.to_numpy(np.int32)]

    def rows(self, codes):
        try:
            return np.asarray([self.lookup[int(c)] for c in codes], np.int32)
        except KeyError as exc:
            raise ValueError(f"Article absent from catalogue: {exc}") from exc

    def eligible(self, codes, t):
        return (
            self.availability.loc[list(codes), "available_from"] <= pd.Timestamp(t)
        ).to_numpy(bool)

### Historical state and user profiles

In [ ]:
HOUR = 3600000000000

In [ ]:
class PairStore:
    """Disk-backed user/article history; avoids a 97M-row Python dictionary.

    A fresh replay must use a fresh database. The persistent database is a scratch
    backend, not a complete replay checkpoint (profile/article state is separate).
    """

    def __init__(self, path=":memory:"):
        if str(path) != ":memory:":
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            if path.exists():
                raise FileExistsError(
                    f"Pair state already exists: {path}. Use a fresh replay path."
                )
        self.db = sqlite3.connect(str(path))
        self.db.execute("PRAGMA journal_mode=WAL")
        self.db.execute("PRAGMA synchronous=NORMAL")
        self.db.execute("PRAGMA cache_size=-32768")
        self.db.execute(
            "CREATE TABLE pairs(u INTEGER,a INTEGER,e INTEGER,c INTEGER,last INTEGER,PRIMARY KEY(u,a)) WITHOUT ROWID"
        )

    def lookup(self, users, articles):
        result = []
        pairs = list(zip(map(int, users), map(int, articles)))
        for start in range(0, len(pairs), 300):
            chunk = pairs[start : start + 300]
            sql = (
                "WITH wanted(i,u,a) AS (VALUES "
                + ",".join(["(?,?,?)"] * len(chunk))
                + ") SELECT COALESCE(p.e,0),COALESCE(p.c,0),p.last FROM wanted w LEFT JOIN pairs p ON p.u=w.u AND p.a=w.a ORDER BY w.i"
            )
            result.extend(
                self.db.execute(
                    sql, [v for i, pair in enumerate(chunk) for v in (i, *pair)]
                ).fetchall()
            )
        return result

    def update(self, users, articles, labels, t):
        grouped = Counter()
        for u, a, y in zip(users, articles, labels):
            grouped[int(u), int(a), "e"] += 1
            grouped[int(u), int(a), "c"] += int(y)
        keys = {(u, a) for u, a, _ in grouped}
        rows = [
            (u, a, grouped[u, a, "e"], grouped[u, a, "c"], int(t.value))
            for u, a in sorted(keys)
        ]
        self.db.executemany(
            "INSERT INTO pairs VALUES(?,?,?,?,?) ON CONFLICT(u,a) DO UPDATE SET e=e+excluded.e,c=c+excluded.c,last=excluded.last",
            rows,
        )
        self.db.commit()

    def close(self):
        self.db.close()

In [ ]:
@dataclass
class User:
    snapshot: np.ndarray
    snapshot_key: tuple
    observed: deque
    categories: Counter = field(default_factory=Counter)
    subcategories: Counter = field(default_factory=Counter)
    n_observed: int = 0
    last_impression: pd.Timestamp | None = None
    version: int = 0

In [ ]:
class TemporalState:
    def __init__(self, bundle, start, end, cfg=DEFAULT_SETTINGS, pair_path=":memory:"):
        self.bundle = bundle
        self.cfg = cfg
        self.start = pd.Timestamp(start)
        self.end = pd.Timestamp(end)
        self.start_hour = self.start.value // HOUR
        hours = int(self.end.value // HOUR - self.start_hour) + 1
        if hours < 1:
            raise ValueError("End before start")
        self.exposures = np.zeros(bundle.n, np.int64)
        self.clicks = np.zeros(bundle.n, np.int64)
        self.first = np.full(bundle.n, -1, np.int64)
        self.hour_exposures = np.zeros((bundle.n, hours), np.int32)
        self.hour_clicks = np.zeros((bundle.n, hours), np.int32)
        self.cat = np.concatenate(
            [
                ["<PAD>"],
                bundle.index.category.fillna("<MISSING>").astype(str).to_numpy(),
            ]
        )
        self.subcat = np.concatenate(
            [
                ["<PAD>"],
                bundle.index.subcategory.fillna("<MISSING>").astype(str).to_numpy(),
            ]
        )
        self.users = {}
        self.profiles = {}
        self.pairs = PairStore(pair_path)
        self.total_exposures = 0
        self.total_clicks = 0
        self.last_update = None
        self.active_time = None

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple((int(token.removeprefix("N")) for token in text.split()))
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=self.cfg.history_cap))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.sort(user.snapshot)
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        text, valid_text = normalised_mean(self.bundle.text, history)
        entity, valid_entity = normalised_mean(self.bundle.entity, history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            rows = history[self.cat[history] == cat]
            vector, valid = normalised_mean(self.bundle.title, rows)
            if valid and np.linalg.norm(vector) > 0:
                interests.append(vector)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(history)),
            "valid_entity_fraction": valid_entity / max(1, len(history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def half_life(self, rows, t):
        t = pd.Timestamp(t)
        recent_end = int(t.value // HOUR - self.start_hour)
        recent_start = recent_end - 6
        result = []
        for row in rows:
            first = self.first[row]
            if first < 0:
                result.append((0.0, False, "no_prior_exposure", 0, 0))
                continue
            early_start = int((first + HOUR - 1) // HOUR - self.start_hour)
            early_end = early_start + 6
            if recent_start < early_end:
                result.append((0.0, False, "young_or_overlapping", 0, 0))
                continue
            early_e = int(self.hour_exposures[row, early_start:early_end].sum())
            recent_e = int(self.hour_exposures[row, recent_start:recent_end].sum())
            if min(early_e, recent_e) < self.cfg.half_life_min_exposures:
                result.append((0.0, False, "low_support", early_e, recent_e))
                continue
            early_c = int(self.hour_clicks[row, early_start:early_end].sum())
            recent_c = int(self.hour_clicks[row, recent_start:recent_end].sum())
            early = (early_c + 1) / (early_e + 2)
            recent = (recent_c + 1) / (recent_e + 2)
            if recent >= early:
                result.append((0.0, False, "no_decline", early_e, recent_e))
                continue
            estimate = np.log(2) * (recent_start - early_start) / np.log(early / recent)
            ok = bool(np.isfinite(estimate) and estimate > 0)
            result.append(
                (
                    float(estimate) if ok else 0.0,
                    ok,
                    "available" if ok else "invalid",
                    early_e,
                    recent_e,
                )
            )
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

    def close(self):
        self.pairs.close()

### Reading and writing the datasets

In [ ]:
def timestamp_groups(path, columns, batch_size=100000):
    """Bounded parquet reads retaining complete timestamp groups across row groups."""
    pending = None
    last = None
    for batch in pq.ParquetFile(path).iter_batches(
        batch_size=batch_size, columns=columns
    ):
        frame = batch.to_pandas()
        frame["timestamp"] = pd.to_datetime(frame.timestamp).astype("datetime64[ns]")
        if frame.timestamp.isna().any() or not frame.timestamp.is_monotonic_increasing:
            raise ValueError(f"Unsorted or missing timestamps: {path}")
        if last is not None and frame.timestamp.iloc[0] < last:
            raise ValueError(f"Parquet row groups out of order: {path}")
        last = frame.timestamp.iloc[-1]
        if pending is not None:
            frame = pd.concat([pending, frame], ignore_index=True)
        tail = frame.timestamp.iloc[-1]
        ready = frame[frame.timestamp != tail]
        pending = frame[frame.timestamp == tail]
        for t, group in ready.groupby("timestamp", sort=False):
            yield (pd.Timestamp(t), group.reset_index(drop=True))
    if pending is not None:
        yield (pd.Timestamp(pending.timestamp.iloc[0]), pending.reset_index(drop=True))

In [ ]:
def replay_batches(root, cfg=DEFAULT_SETTINGS, splits=SPLITS):
    processed = require_inputs(root)
    streams = []
    for split in splits:
        for kind, columns in (
            ("impressions", ["impression_uid", "user_code", "timestamp", "history"]),
            (
                "candidates",
                ["impression_uid", "user_code", "timestamp", "news_code", "clicked"],
            ),
        ):
            streams.append(
                (
                    kind,
                    split,
                    iter(
                        timestamp_groups(
                            processed / f"{kind}_{split}.parquet",
                            columns,
                            cfg.parquet_batch_rows,
                        )
                    ),
                )
            )
    heap = []
    for i, (_, _, stream) in enumerate(streams):
        first = next(stream, None)
        if first is not None:
            heapq.heappush(heap, (first[0], i, first[1]))
    while heap:
        t = heap[0][0]
        requests = []
        events = []
        while heap and heap[0][0] == t:
            _, i, frame = heapq.heappop(heap)
            kind, split, stream = streams[i]
            frame = frame.copy()
            frame["split"] = split
            (requests if kind == "impressions" else events).append(frame)
            nxt = next(stream, None)
            if nxt is not None:
                heapq.heappush(heap, (nxt[0], i, nxt[1]))
        if not requests or not events:
            raise ValueError(f"Unpaired requests/events at {t}")
        req = pd.concat(requests, ignore_index=True)
        ev = pd.concat(events, ignore_index=True)
        if (
            req.impression_uid.duplicated().any()
            or ev.duplicated(["impression_uid", "news_code"]).any()
        ):
            raise ValueError("Duplicate request/pair in timestamp stream")
        if set(ev.impression_uid) != set(req.impression_uid):
            raise ValueError("Every request must have its logged candidate slate")
        owners = req.set_index("impression_uid").user_code
        if not np.array_equal(
            ev.user_code.to_numpy(), ev.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Candidate/request user mismatch")
        yield (t, req, ev)

In [ ]:
def time_horizon(root, splits=SPLITS):
    import polars as pl

    p = require_inputs(root)
    ranges = [
        pl.scan_parquet(p / f"impressions_{s}.parquet")
        .select(
            pl.col("timestamp").min().alias("min"),
            pl.col("timestamp").max().alias("max"),
        )
        .collect()
        .row(0)
        for s in splits
    ]
    return (min((x[0] for x in ranges)), max((x[1] for x in ranges)))

In [ ]:
def build_supplied_profiles(root, cfg=DEFAULT_SETTINGS):
    """Supplied-only matrix for baseline/cluster handoff; no in-window feedback."""
    processed = require_inputs(root)
    out = artifact_dir(root)
    bundle = Bundle(root, cfg)
    with measure(root, "supplied_profiles"):
        users = (
            pd.read_parquet(processed / "users.parquet")
            .sort_values("user_code")
            .reset_index(drop=True)
        )
        if users.user_code.duplicated().any():
            raise ValueError("Duplicate user codes")
        users["profile_row"] = np.arange(len(users), dtype=np.int32)
        user_rows = dict(zip(users.user_code, users.profile_row))
        text = np.lib.format.open_memmap(
            out / "supplied_text_profiles.npy",
            mode="w+",
            dtype=np.float32,
            shape=(len(users), cfg.text_dim),
        )
        entity = np.lib.format.open_memmap(
            out / "supplied_entity_profiles.npy",
            mode="w+",
            dtype=np.float32,
            shape=(len(users), cfg.entity_dim),
        )
        text[:] = 0
        entity[:] = 0
        quality = []
        seen = {}
        train_users = set()
        for split in SPLITS:
            for batch in pq.ParquetFile(
                processed / f"impressions_{split}.parquet"
            ).iter_batches(
                batch_size=cfg.parquet_batch_rows, columns=["user_code", "history"]
            ):
                for row in (
                    batch.to_pandas()
                    .drop_duplicates("user_code")
                    .itertuples(index=False)
                ):
                    u = int(row.user_code)
                    if split == "train":
                        train_users.add(u)
                    raw = "" if pd.isna(row.history) else str(row.history)
                    if u in seen:
                        if seen[u] != raw:
                            raise ValueError(f"Supplied history changed: {u}")
                        continue
                    seen[u] = raw
                    codes = [int(x.removeprefix("N")) for x in raw.split()]
                    rows = bundle.rows(codes)
                    i = user_rows[u]
                    text[i], vt = normalised_mean(bundle.text, rows)
                    entity[i], ve = normalised_mean(bundle.entity, rows)
                    quality.append(
                        {
                            "user_code": u,
                            "history_length": len(rows),
                            "valid_text_history_count": vt,
                            "valid_entity_history_count": ve,
                            "history_truncated": len(rows) > cfg.history_cap,
                        }
                    )
        if len(seen) != len(users):
            raise ValueError("Users mapping does not match observed users")
        text.flush()
        entity.flush()
        users = users.merge(
            pd.DataFrame(quality), on="user_code", validate="one_to_one", sort=False
        )
        users["seen_in_train"] = users.user_code.isin(train_users)
        users["has_text_profile"] = np.linalg.norm(text, axis=1) > 0
        users["has_entity_profile"] = np.linalg.norm(entity, axis=1) > 0
        users.to_parquet(out / "supplied_profile_index.parquet", index=False)
        coverage = []
        for name, count in (
            ("supplied_text_profile", "valid_text_history_count"),
            ("supplied_entity_profile", "valid_entity_history_count"),
        ):
            for cohort, mask in (
                ("all_users", np.ones(len(users), bool)),
                ("training_users", users.seen_in_train.to_numpy()),
            ):
                n = int(mask.sum())
                k = int(
                    users.loc[
                        mask,
                        "has_text_profile"
                        if count == "valid_text_history_count"
                        else "has_entity_profile",
                    ].sum()
                )
                coverage.append(
                    {
                        "split": "all" if cohort == "all_users" else "train",
                        "representation": name,
                        "cohort": cohort,
                        "denominator": n,
                        "valid": k,
                        "fallback": n - k,
                        "valid_fraction": k / max(1, n),
                    }
                )
        upsert_report(
            root,
            "representation_coverage.csv",
            pd.DataFrame(coverage),
            "supplied_profiles",
        )
        files = [
            out / f
            for f in (
                "supplied_text_profiles.npy",
                "supplied_entity_profiles.npy",
                "supplied_profile_index.parquet",
            )
        ]
        sources = [processed / f"impressions_{s}.parquet" for s in SPLITS] + [
            out / "news_manifest.json",
            out / "entity_manifest.json",
        ]
        write_json(
            out / "profiles_manifest.json",
            manifest(
                root,
                cfg,
                sources,
                files,
                clustering_policy="fit scaler/clusters only on seen_in_train rows; supplied history only",
                temporal_policy="static supplied profiles are not time-updated ranking profiles",
                article_index_hash=sha256(out / "article_index.parquet"),
            ),
        )
    return users

In [ ]:
def saved_stage_ready(root, settings, stage):
    """Reuse disk outputs only when settings, inputs and output hashes agree."""
    path = Path(root) / "3_Representations/artifacts" / f"{stage}_manifest.json"
    if not path.exists():
        return False
    meta = json.loads(path.read_text())
    key = "config_hash" if stage == "profiles" else "representation_config_hash"
    expected = (
        settings.fingerprint()
        if stage == "profiles"
        else settings.representation_fingerprint()
    )
    if meta[key] != expected:
        raise ValueError(
            "Saved settings differ. Set REBUILD=True and rerun dependent notebooks."
        )
    for kind in ("sources", "outputs"):
        for relative, expected_hash in meta[kind].items():
            file = Path(root) / relative
            if not file.exists() or sha256(file) != expected_hash:
                raise ValueError(
                    f"Saved {stage} artifacts are stale or incomplete: {relative}. Set REBUILD=True."
                )
    return True

## Build or load saved outputs

## 1. Observation-time eligibility

`available_from` is the earliest observable slate appearance or supplied-history discovery, not publication time.
Do not backdate new test users' histories to training start. A history-only article has no observed exposure age.
Eligibility uses `available_from <= t`; it is separate from counts built from events strictly before `t`.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "availability"):
    availability = build_availability(ROOT, SETTINGS)
else:
    print("Loaded saved article availability.")
    availability = pd.read_parquet(ARTIFACTS / "article_availability.parquet")
display(availability.head())
display(availability.discovery_source.value_counts())

## 2. Supplied-history profiles

Pool every valid supplied-history article vector (no time decay because click times are unknown).
These matrices never consume logged clicks. They are appropriate for the report's clustering experiment:
fit a scaler/K-means only on rows with `seen_in_train=True`; freeze both for later users.
A user absent from training can still have a usable supplied history.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "profiles"):
    profiles = build_supplied_profiles(ROOT, SETTINGS)
else:
    print("Loaded saved supplied-history profiles.")
    profiles = pd.read_parquet(ARTIFACTS / "supplied_profile_index.parquet")
display(
    profiles.groupby("seen_in_train").agg(
        users=("user_code", "size"),
        no_history=("history_length", lambda x: int(x.eq(0).sum())),
        no_entity_profile=("valid_entity_history_count", lambda x: int(x.eq(0).sum())),
    )
)
display(profiles.head())

## 3. Demonstrate causal profile updates

Seed each user once; append genuine earlier click events, retaining repeats. Score all requests in a timestamp
batch before consuming its feedback. Full category counts use all prior clicks; vector/NRMS histories are capped.
Snapshot chronology is unverified by default, so its capped subset is deterministic and not called recent.
Recent profiles use timestamped in-window clicks only. Up to three category interests are ranked by full
prior category counts and pooled from valid capped title vectors. Empty groups are masked.

The example processes only three initial timestamp batches, with their preceding state intact.

In [ ]:
from itertools import islice

bundle = Bundle(ROOT, SETTINGS)
state = TemporalState(bundle, *time_horizon(ROOT), SETTINGS)
examples = []
try:
    for t, requests, events in islice(replay_batches(ROOT, SETTINGS), 3):
        state.begin(requests, t)
        for request in requests.head(2).itertuples(index=False):
            profile = state.profile(request.user_code, t)
            examples.append(
                dict(
                    timestamp=t,
                    impression_uid=request.impression_uid,
                    user_code=request.user_code,
                    history_length=profile["history_length"],
                    retained_length=profile["retained_length"],
                    valid_interests=profile["group_count"],
                    has_text=profile["has_text"],
                    has_entity=profile["has_entity"],
                    has_recent=profile["has_recent"],
                )
            )
        state.finish(requests, events, t, {"train", "validation", "test"})
finally:
    state.close()
display(pd.DataFrame(examples))
display(pd.read_csv(ROOT / "results/representation_coverage.csv"))

## Handoff

The `TemporalState` class defined above exposes causal text/entity/recent profiles, category counts, interest vectors,
and padded history-row sequences/masks. The feature build writes per-request history references so
retrieval, attention rankers and future ablations can reconstruct the same profile. State is timestamp-scoped;
a final user vector must never be reused for an earlier request. The disk pair store is replay scratch space,
not a resumable checkpoint. Persisted resume support is not implemented in this version.